# SAMURAI Optimize — VS Code Notebook (2-venv workflow)

Notebook để so sánh SAMURAI **gốc** vs **optimized** trên LaSOT-small.

## Workflow

Mỗi bản `sam2` có **venv riêng** → không conflict distribution, **không cần restart kernel** khi chuyển bản. Chỉ cần **Select Kernel** chọn đúng venv.

```
samurai_optimized/
├── .venv/              ← venv cho bản OPTIMIZED
├── sam2/               ← fork optimized (pip install -e vào .venv)
├── scripts/
├── samurai/
│   ├── .venv/          ← venv cho bản GỐC
│   ├── sam2/           ← SAMURAI gốc (pip install -e vào samurai/.venv)
│   └── scripts/
```

**Quy tắc:**
1. Chọn đúng kernel trước khi chạy section A hoặc B.
   - Chạy bản gốc → kernel `samurai/.venv/bin/python`
   - Chạy bản optimized → kernel `.venv/bin/python`
2. **Luôn chạy Cell 0 đầu tiên** sau mỗi lần đổi kernel.
3. Dùng GPU **hỗ trợ PyTorch stable** (T4, A100, L4, RTX 30xx/40xx). RTX 5080/5090 (Blackwell sm_120) chưa có PyTorch prebuilt.

## 0. Cấu hình đường dẫn + fix sys.path

In [ ]:
import os, sys


_cwd = os.path.abspath(os.getcwd())
sys.path[:] = [p for p in sys.path
               if p != '' and os.path.abspath(p) != _cwd]
print("Removed CWD from sys.path:", _cwd)


REPO_ROOT   = "/home/ubuntu-phuocbh/Downloads/Khoa_luan_tot_nghiep_sam2/samurai_optimized"
DATA_ROOT   = "/path/to/lasot-small/small_LaSOT"
TESTING_SET = os.path.join(REPO_ROOT, "testing_set.txt")

os.environ["REPO_ROOT"]   = REPO_ROOT
os.environ["DATA_ROOT"]   = DATA_ROOT
os.environ["TESTING_SET"] = TESTING_SET

print("REPO_ROOT  =", REPO_ROOT)
print("DATA_ROOT  =", DATA_ROOT)
print("TESTING_SET=", TESTING_SET)
print("Python     =", sys.executable)

## 1. Tạo 2 venv (chạy 1 lần duy nhất)

Chạy trong terminal hoặc chạy các cell dưới.

> Nếu đã tạo venv rồi thì **bỏ qua** section này.

### 1.0 — Auto-detect GPU → chọn PyTorch index URL

In [ ]:
import subprocess, os

# Detect GPU compute capability via nvidia-smi
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu121"

try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
        text=True
    ).strip().split('\n')[0]
    major, minor = map(int, out.split('.'))
    cc = major * 10 + minor
    print(f'GPU compute capability: sm_{major}{minor} ({out})')

    if cc >= 120:
        # Blackwell (RTX 5080/5090)
        print('⚠ Blackwell GPU (sm_120+) — PyTorch prebuilt CHƯA hỗ trợ.')
        print('  Dùng GPU khác (T4, A100, L4, RTX 30xx/40xx) hoặc build from source.')
        TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu124'
    elif cc >= 90:
        # Hopper (H100/H200)
        TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu124'
        print('→ Hopper GPU → dùng PyTorch stable cu124')
    else:
        # Ampere, Turing, Pascal, etc.
        TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu121'
        print('→ dùng PyTorch stable cu121')
except Exception as e:
    print(f'Không detect được GPU: {e}')
    print('→ Fallback: dùng PyTorch stable cu121')

os.environ['TORCH_INDEX_URL'] = TORCH_INDEX_URL
print(f'TORCH_INDEX_URL = {TORCH_INDEX_URL}')

### 1a. Venv cho bản OPTIMIZED (`samurai_optimized/.venv`)

In [ ]:
%%bash
set -e
cd "$REPO_ROOT"

[ -d .venv ] || python3 -m venv .venv
source .venv/bin/activate

pip install --upgrade pip setuptools wheel
echo "Installing torch from: $TORCH_INDEX_URL"
pip install torch torchvision --index-url "$TORCH_INDEX_URL"

SAM2_BUILD_CUDA=0 pip install --no-build-isolation -e sam2/

pip install matplotlib==3.7 tikzplotlib jpeg4py opencv-python lmdb pandas scipy loguru psutil

echo "=== DONE: venv optimized ==="
which python
python -c "import torch; print('torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"

cd /tmp && python -c "import sam2; print('sam2:', sam2.__file__)" 

### 1b. Venv cho bản GỐC (`samurai_optimized/samurai/.venv`)

In [ ]:
%%bash
set -e
cd "$REPO_ROOT/samurai"

[ -d .venv ] || python3 -m venv .venv
source .venv/bin/activate

pip install --upgrade pip setuptools wheel
echo "Installing torch from: $TORCH_INDEX_URL"
pip install torch torchvision --index-url "$TORCH_INDEX_URL"

SAM2_BUILD_CUDA=0 pip install --no-build-isolation -e sam2/

pip install matplotlib==3.7 tikzplotlib jpeg4py opencv-python lmdb pandas scipy loguru psutil

echo "=== DONE: venv baseline ==="
which python
python -c "import torch; print('torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"
cd /tmp && python -c "import sam2; print('sam2:', sam2.__file__)" 

### 1c. Đăng ký kernel cho VS Code

Sau khi tạo xong 2 venv:
1. **Command Palette** (`Ctrl+Shift+P`) → `Python: Select Interpreter` → `Enter interpreter path...`
2. Trỏ lần lượt tới:
   - `samurai_optimized/.venv/bin/python`  (optimized)
   - `samurai_optimized/samurai/.venv/bin/python`  (baseline)
3. Khi mở notebook, click **Select Kernel** góc trên-phải sẽ thấy cả 2 venv.

> Verify bằng `import sys; print(sys.executable)` — phải ra đúng path venv.

## 2. Download checkpoints (chạy 1 lần)

In [ ]:
# Checkpoints cho bản optimized
!cd "$REPO_ROOT/sam2/checkpoints" && bash download_ckpts.sh

In [ ]:
# Checkpoints cho bản gốc
!cd "$REPO_ROOT/samurai/sam2/checkpoints" && bash download_ckpts.sh

## 3. Tạo testing set

In [ ]:
import os
os.makedirs(os.path.dirname(TESTING_SET) or ".", exist_ok=True)
with open(TESTING_SET, "w") as f:
    f.write("mouse-1\nelectricfan-1\ngecko-1\n")
print("Wrote", TESTING_SET)
print(open(TESTING_SET).read())

---

## A. Chạy bản SAMURAI **GỐC**

### Chọn kernel: `samurai/.venv/bin/python`

Click **Select Kernel** → chọn interpreter `samurai_optimized/samurai/.venv/bin/python`.

Sau khi chọn xong, **chạy lại Cell 0** rồi tiếp tục.

In [ ]:
# A.1 — Verify đúng venv baseline
import sys, sam2, torch
print("Python    :", sys.executable)
print("sam2 file :", sam2.__file__)
print("torch     :", torch.__version__)
print("CUDA      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))

assert sam2.__file__ is not None, \
    "sam2.__file__ = None → namespace fallback. Chạy lại Cell 0 (fix sys.path) rồi restart kernel."
assert "samurai/sam2" in sam2.__file__.replace("\\", "/"), \
    f"Load nhầm bản: {sam2.__file__}. Phải chứa 'samurai/sam2'. Đã chọn đúng kernel?"
assert torch.cuda.is_available(), \
    "CUDA not available. Check GPU và torch version."
print("\nOK — đang dùng bản GỐC")

### A.2 — Chạy inference baseline

In [ ]:
# Dùng sys.executable để chạy đúng python của kernel/venv
# (bare !python sẽ gọi python hệ thống qua PATH → sai venv)
!{sys.executable} "$REPO_ROOT/samurai/scripts/main_inference.py" \
    --model_name base_plus \
    --evaluate \
    --data_root "$DATA_ROOT" \
    --testing_set "$TESTING_SET" 

---

## B. Chạy bản **OPTIMIZED**

### Chọn kernel: `.venv/bin/python`

Click **Select Kernel** → chọn interpreter `samurai_optimized/.venv/bin/python`.

Sau khi chọn xong, **chạy lại Cell 0** rồi tiếp tục.

In [ ]:
# B.1 — Verify đúng venv optimized
import sys, sam2, torch
print("Python    :", sys.executable)
print("sam2 file :", sam2.__file__)
print("torch     :", torch.__version__)
print("CUDA      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))

assert sam2.__file__ is not None, \
    "sam2.__file__ = None → namespace fallback. Chạy lại Cell 0 (fix sys.path) rồi restart kernel."
p = sam2.__file__.replace("\\", "/")
assert "/samurai/" not in p and "samurai_optimized/sam2" in p, \
    f"Load nhầm bản: {sam2.__file__}. Phải trỏ tới samurai_optimized/sam2/. Đã chọn đúng kernel?"
assert torch.cuda.is_available(), \
    "CUDA not available. Check GPU và torch version."
print("\nOK — đang dùng bản OPTIMIZED")

### B.2 — Chạy inference optimized

Flag `--optimized` **bắt buộc**, nếu không `release_interval=0` → bypass toàn bộ logic tối ưu.

In [ ]:
!{sys.executable} "$REPO_ROOT/scripts/main_inference.py" \
    --optimized \
    --model_name base_plus \
    --log_metrics \
    --run_tag optimized \
    --evaluate \
    --data_root "$DATA_ROOT" \
    --testing_set "$TESTING_SET" 

Các flag tối ưu có thể tinh chỉnh:

| Flag | Default | Ý nghĩa |
|---|---:|---|
| `--release_interval` | 60 | Mỗi N frame giải phóng frame cũ |
| `--keep_window_maskmem` | 1000 | Số frame giữ maskmem_features |
| `--keep_window_pred_masks` | 60 | Số frame giữ pred_masks |
| `--max_cache_frames` | 60 | LRU cache images trong RAM |
| `--no_auto_promote` | off | Tắt auto-promote cond frames |

---

## C. So sánh kết quả

Chạy cell này với **kernel nào cũng được** (chỉ cần matplotlib + pandas).

In [ ]:
!{sys.executable} "$REPO_ROOT/scripts/plot_metrics.py" \
    --run "$REPO_ROOT/metrics/samurai_base_plus/baseline" \
    --run "$REPO_ROOT/metrics/samurai_base_plus/optimized" \
    --label Baseline --label Optimized \
    --mode per_video

---

## Troubleshooting

| Triệu chứng | Nguyên nhân | Khắc phục |
|---|---|---|
| `sam2.__file__ = None` | Thư mục `sam2/` ở CWD shadow installed package | Chạy lại **Cell 0** (loại CWD khỏi sys.path) |
| `CUDA not available` | Kernel dùng python hệ thống thay vì venv | **Select Kernel** → chọn đúng `.venv/bin/python` |
| `no kernel image for sm_XXX` | GPU không nằm trong danh sách arch | Đổi GPU hoặc đổi torch version |
| Metrics 2 run trùng khít | Quên `--optimized` hoặc cùng `sam2` package | Verify `sam2.__file__` + flag `--optimized` |
| `pip install -e sam2/` fail | Build isolation tải torch mới từ PyPI | Dùng `--no-build-isolation` (đã có trong notebook) |
| `!python` chạy sai interpreter | `!python` dùng PATH hệ thống | Notebook đã dùng `{sys.executable}` |